In [ ]:
%%html
<style>
    body {
        --vscode-font-family: "Ricty";
    }
</style>

In [ ]:
import hashlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from tqdm import tqdm

In [ ]:
URL_CLUSTER_TRIAL = "https://raw.githubusercontent.com/HirotakeIto/intro_to_impact_evaluation_with_python/main/data/ch3_cluster_trial.csv"

In [ ]:
# ------------------------------------------------------------------------
# クラスターA/Bテストのデータに対して、A/Aテストのリプレイ
# ------------------------------------------------------------------------

In [ ]:
def assign_treatment_randomly(imp_id, salt):
    return int(hashlib.sha256(f"{salt}_{imp_id}".encode()).hexdigest(), 16) % 2

In [ ]:
aatest_df = pd.read_csv(URL_CLUSTER_TRIAL)
aatest_df.head()

In [ ]:
rng = np.random.default_rng(seed=0)

In [ ]:
replays = []
for i in tqdm(range(300)):
    salt = f"salt{i}"
    aatest_df["is_treatment_in_aa"] = aatest_df["uid"].apply(
        assign_treatment_randomly, salt=salt
    )
    # 回帰分析
    result = smf.ols("is_click ~ is_treatment_in_aa", data=aatest_df).fit()
    pvalue = result.pvalues["is_treatment_in_aa"]
    replays.append(pvalue)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.hist(replays)
ax.set_facecolor("none")
ax.set_xlabel("p-value")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of p-values in A/A test replays")
plt.show()

In [ ]:
# ------------------------------------------------------------------------
# コルモゴロフ・スミルノフ検定による分布の検定
# ------------------------------------------------------------------------

In [ ]:
stats.kstest(replays, "uniform", args=(0, 1))